# SIN Assistant
## Backend RAG para el Sistema Interconectado Nacional de Colombia

# 1. Importación de librerías
En esta sección se importan las librerías necesarias para la extracción de información, generación de embeddings y almacenamiento vectorial.

In [51]:
import os
import logging
import requests
import chromadb

from bs4 import BeautifulSoup
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# 2. Extracción del glosario XM

En esta sección se descarga el contenido del glosario público de XM, utilizado como base de conocimiento para SIN Assistant.

In [52]:
url = "https://www.xm.com.co/herramientas/glosario-xm"

html = requests.get(url).text

print("Estado:", requests.get(url).status_code)
print("Caracteres:", len(html))

Estado: 200
Caracteres: 419206


# 3. Limpieza del texto

El contenido descargado contiene elementos de navegación, menús y otra información no relacionada con el glosario. En esta sección se extrae el texto relevante y se genera una versión limpia para su procesamiento posterior.

In [53]:
soup = BeautifulSoup(html, "html.parser")

texto = soup.get_text(" ", strip=True)

inicio = texto.find("Acción de tutela:")

import re

glosario_limpio = texto[inicio:]

glosario_limpio = re.sub(
    r'(?<!^)([A-ZÁÉÍÓÚÑ][A-Za-zÁÉÍÓÚÑáéíóúñ0-9 ()/-]{3,100}:)',
    r'\n\n\1',
    glosario_limpio
)

print("Caracteres glosario limpio:", len(glosario_limpio))

Caracteres glosario limpio: 81124


In [54]:
print(glosario_limpio[:3000])

Acción de tutela: Se encuentra consagrada en el artículo 86 de la Constitución Nacional. Es la acción que tiene toda persona natural o jurídica como mecanismo preferencial frente a la ausencia de otro procedimiento, para reclamar ante las autoridades judiciales la protección inmediata de sus derechos constitucionales fundamentales, cuando quiera que estos resulten vulnerados o amenazados por la acción u omisión de cualquier autoridad pública. Su trámite procedimental se encuentra reglamentado en el Decreto 2591 de 1991, que a su vez fue reglamentado por el Decreto 306 de 1992; así como en la Resolución 669 de 2000 expedida por la Defensoría del Pueblo. 

Actividad de comercialización de energía eléctrica: Actividad consistente en la compra de energía eléctrica en el mercado mayorista y su venta a los usuarios finales, bien sea que esa actividad se desarrolle o no en forma combinada con otras actividades del sector eléctrico, cualquiera sea la actividad principal. (Art. 11, Ley 143 de 1

# 4. Almacenamiento del glosario limpio

Una vez extraído el contenido relevante del glosario, se almacena localmente para su posterior utilización en el sistema RAG.

In [55]:
ruta_salida = r"C:\Users\crist\OneDrive\CRISTHIAN CRISTANCHO\Ciencia de Datos y Analítica\6to Semestre\NLP\Phase 4\SIN_Assistant\knowledge_base\processed\xm_glossary_clean.txt"

with open(ruta_salida, "w", encoding="utf-8") as archivo:
    archivo.write(glosario_limpio)

print("Archivo guardado correctamente")
print("Caracteres:", len(glosario_limpio))

Archivo guardado correctamente
Caracteres: 81124


# 5. Segmentación del documento

El glosario limpio se divide en fragmentos de tamaño fijo para facilitar el proceso de recuperación semántica dentro del sistema RAG.

In [56]:
import re

conceptos = [
    concepto.strip()
    for concepto in glosario_limpio.split("\n\n")
    if concepto.strip()
]

conceptos_mejorados = []

for concepto in conceptos:

    siglas = re.findall(r'\(([A-Z]{2,10})\)', concepto)

    if siglas:
        encabezado = " ".join(siglas)
        concepto = f"{encabezado}\n\n{concepto}"

    conceptos_mejorados.append(concepto)

conceptos = conceptos_mejorados

print("Cantidad de conceptos:", len(conceptos))

Cantidad de conceptos: 329


In [59]:
print(conceptos[0])

Acción de tutela: Se encuentra consagrada en el artículo 86 de la Constitución Nacional. Es la acción que tiene toda persona natural o jurídica como mecanismo preferencial frente a la ausencia de otro procedimiento, para reclamar ante las autoridades judiciales la protección inmediata de sus derechos constitucionales fundamentales, cuando quiera que estos resulten vulnerados o amenazados por la acción u omisión de cualquier autoridad pública. Su trámite procedimental se encuentra reglamentado en el Decreto 2591 de 1991, que a su vez fue reglamentado por el Decreto 306 de 1992; así como en la Resolución 669 de 2000 expedida por la Defensoría del Pueblo.


# 6. Generación de embeddings

Los fragmentos generados se transforman en representaciones vectoriales utilizando el modelo abierto Sentence-Transformers all-MiniLM-L6-v2. Estos vectores permiten realizar búsquedas semánticas dentro de la base de conocimiento.

In [60]:
modelo_embeddings = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5081.61it/s]


In [61]:
embeddings = modelo_embeddings.encode(
    conceptos,
    show_progress_bar=True
)

print("Cantidad de embeddings:", len(embeddings))
print("Dimensión del primer embedding:", len(embeddings[0]))

Batches: 100%|██████████| 11/11 [00:02<00:00,  4.21it/s]

Cantidad de embeddings: 329
Dimensión del primer embedding: 384


# 7. Base vectorial ChromaDB

Los fragmentos generados y sus embeddings se almacenan en una base vectorial persistente utilizando ChromaDB. Esto permite recuperar contexto relevante para responder preguntas de los usuarios.

In [62]:
cliente = chromadb.PersistentClient(
    path=r"C:\Users\crist\OneDrive\CRISTHIAN CRISTANCHO\Ciencia de Datos y Analítica\6to Semestre\NLP\Phase 4\SIN_Assistant\chroma_db"
)

try:
    cliente.delete_collection("xm_glossary")
    print("Colección anterior eliminada")
except:
    print("No existía una colección previa")

coleccion = cliente.get_or_create_collection(
    name="xm_glossary"
)

print("Base vectorial creada correctamente")


Colección anterior eliminada
Base vectorial creada correctamente


In [63]:
coleccion.add(
    documents=conceptos,
    ids=[f"concepto_{i}" for i in range(len(conceptos))]
)

print("Chunks almacenados en ChromaDB")

Chunks almacenados en ChromaDB


# 8. Recuperación de información

Se realiza una consulta semántica sobre la base vectorial para verificar que el sistema puede recuperar información relevante del glosario XM.

In [82]:
pregunta = "¿Qué es el SDL?"

consulta = pregunta.upper()

consulta = consulta.replace("¿QUÉ ES EL ", "")
consulta = consulta.replace("¿QUE ES EL ", "")
consulta = consulta.replace("?", "")
consulta = consulta.strip()

print("Consulta utilizada:", consulta)

resultado = coleccion.query(
    query_texts=[consulta],
    n_results=5
)

Consulta utilizada: SDL


In [83]:
for i, doc in enumerate(resultado["documents"][0]):
    print(f"\n--- Resultado {i+1} ---\n")
    print(doc[:500])


--- Resultado 1 ---

SDL

Sistema de Distribución Local (SDL): Sistema de transmisión de energía eléctrica compuesto por redes de distribución municipales o distritales; conformado por el conjunto de líneas y subestaciones, con sus equipos asociados, que operan a tensiones menores de 220 kV que no pertenecen a un Sistema de Transmisión Regional por estar dedicadas al servicio de un sistema de distribución municipal, distrital o local.

--- Resultado 2 ---

Activos de uso de STR y SDL: Son aquellos activos de transmisión de electricidad que operan a tensiones inferiores a 220 kV, se clasifican en unidades constructivas, no son activos de conexión, y son remunerados mediante cargos por uso de STR o SDL.

--- Resultado 3 ---

Usuarios de los STR o SDL: Son los usuarios finales del servicio de energía eléctrica, Operadores de Red y generadores conectados a estos sistemas.

--- Resultado 4 ---

Activos de conexión a un STR o a un SDL: Son los bienes que se requieren para que un generador, 

# 9. Ingeniería de prompts

Se define un mensaje del sistema para establecer el comportamiento esperado de SIN Assistant y reducir la generación de información incorrecta.

In [67]:
SYSTEM_PROMPT = """
Eres SIN Assistant.

Eres un asistente especializado en el Sistema Interconectado Nacional (SIN) de Colombia y el Mercado de Energía Mayorista.

REGLAS OBLIGATORIAS:

1. Responde siempre en español.
2. Utiliza únicamente la información recuperada de la base de conocimiento.
3. No inventes información.
4. No uses conocimiento externo.
5. Si la información no aparece en el contexto recuperado, responde exactamente:

   "No encontré suficiente información en la base de conocimiento."

6. Explica los conceptos de forma clara para personas nuevas en la industria eléctrica.
7. Cuando exista una sigla, menciona primero el nombre completo y luego la sigla.
8. Sé breve y preciso.
"""

# 10. Memoria conversacional de corto plazo

Se implementa una memoria básica para almacenar los intercambios recientes entre el usuario y el chatbot. Además, se controla explícitamente la longitud máxima del contexto para evitar un crecimiento indefinido.

In [68]:
conversation_history = []

def save_message(role, message):

    conversation_history.append(
        {
            "role": role,
            "message": message
        }
    )

    MAX_MESSAGES = 10

    if len(conversation_history) > MAX_MESSAGES:
        conversation_history.pop(0)

In [69]:
save_message(
    "user",
    "¿Qué es el ASIC?"
)

save_message(
    "assistant",
    "ASIC es el Administrador del Sistema de Intercambios Comerciales."
)

conversation_history

[{'role': 'user', 'message': '¿Qué es el ASIC?'},
 {'role': 'assistant',
  'message': 'ASIC es el Administrador del Sistema de Intercambios Comerciales.'}]

# 11. Registro de eventos (Logging)

Se implementa un mecanismo básico de registro de eventos para almacenar información útil sobre la ejecución del sistema y las consultas realizadas por los usuarios.

In [70]:
logging.basicConfig(
    filename="chatbot.log",
    level=logging.INFO,
    format="%(asctime)s - %(message)s"
)

logging.info("SIN Assistant iniciado")

In [71]:
logging.info("Consulta: ¿Qué es el ASIC?")

print("Consulta registrada correctamente")

Consulta registrada correctamente


# 12. Integración con un modelo de lenguaje abierto

Se utiliza el modelo abierto Qwen2.5-0.5B-Instruct mediante la biblioteca Transformers. Este modelo es responsable de generar respuestas en lenguaje natural a partir del contexto recuperado por el sistema RAG.

In [72]:
generador = pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-0.5B-Instruct"
)

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 703.80it/s]


In [74]:
respuesta = generador(
    "Responde en español: ¿Qué es el sistema eléctrico?",
    max_new_tokens=60,
    do_sample=False
)

print(respuesta[0]["generated_text"])

Responde en español: ¿Qué es el sistema eléctrico? El sistema eléctrico es una estructura física que permite la distribución y transmisión de energía eléctrica a través de las redes eléctricas. Es un sistema que se utiliza para transportar electricidad de manera eficiente, seguramente más rápida y eficiente que los sistemas


# 13. Generación aumentada por recuperación (RAG)

En esta sección se integra la recuperación semántica mediante ChromaDB con el modelo de lenguaje Qwen. El contexto recuperado se incorpora al prompt antes de generar la respuesta final.

In [75]:
pregunta = "¿Qué es el ASIC?"

consulta = pregunta.upper()

consulta = consulta.replace("¿QUÉ ES EL ", "")
consulta = consulta.replace("¿QUE ES EL ", "")
consulta = consulta.replace("?", "")
consulta = consulta.strip()

resultado = coleccion.query(
    query_texts=[consulta],
    n_results=1
)

contexto = resultado["documents"][0][0]

prompt_rag = f"""
{SYSTEM_PROMPT}

CONTEXTO RECUPERADO:

{contexto}

PREGUNTA DEL USUARIO:

{pregunta}

RESPUESTA:
"""

In [76]:
respuesta_rag = generador(
    prompt_rag,
    max_new_tokens=150,
    do_sample=False
)

print(respuesta_rag[0]["generated_text"])



Eres SIN Assistant.

Eres un asistente especializado en el Sistema Interconectado Nacional (SIN) de Colombia y el Mercado de Energía Mayorista.

REGLAS OBLIGATORIAS:

1. Responde siempre en español.
2. Utiliza únicamente la información recuperada de la base de conocimiento.
3. No inventes información.
4. No uses conocimiento externo.
5. Si la información no aparece en el contexto recuperado, responde exactamente:

   "No encontré suficiente información en la base de conocimiento."

6. Explica los conceptos de forma clara para personas nuevas en la industria eléctrica.
7. Cuando exista una sigla, menciona primero el nombre completo y luego la sigla.
8. Sé breve y preciso.


CONTEXTO RECUPERADO:

ASIC SIC

Administrador del Sistema de Intercambios Comerciales (ASIC): Dependencia del Centro Nacional de Despacho de que tratan las Leyes 142 y 143 de 1994, encargada del registro de fronteras comerciales, de los contratos de energía a largo plazo; de la liquidación, facturación, cobro y pag